# Stacking out-of-fold dan uji berpasangan

[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imamrozi/recsys-hibrida-multisinyal-notebooks/blob/main/notebooks/bab03_sentiment_aware/03_stacking_dan_uji_berpasangan.ipynb)

**Buku pendamping:** *Sistem Rekomendasi Hibrida Multi-Sinyal: Rating, Sentimen, dan Emosi* · Bab 3 · Kode 3.6.1 dan 3.8.1

**Tujuan belajar**

- Memahami mengapa fitur stacking harus berasal dari prediksi out-of-fold.
- Membandingkan representasi sentimen (mean, concat, concat+konfidens) pada fusi pohon keputusan.
- Membaca uji berpasangan, bootstrap berpasangan, dan p gabungan Fisher dengan benar.

> **Catatan jujur.** Semua data di notebook ini **simulasi** dan dibuat agar mudah dijalankan (CPU, hitungan detik). Angka yang keluar menunjukkan *perilaku metode*, bukan hasil empiris pada dataset nyata, dan tidak boleh dikutip sebagai bukti. Kode bersifat ilustratif dan **belum production-ready** (tanpa validasi masukan, tanpa uji beban).

> Seperti notebook sebelumnya, data adalah simulasi dengan asumsi eksplisit. Selisih antar representasi di sini **kecil**, dan justru kasus seperti itu yang membutuhkan uji statistik.

## 1. Kode dari buku: stacking (Kode 3.6.1)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeRegressor


def oof_predictions(fit_predict, train_idx, n_splits=5, seed=0):
    """Prediksi out-of-fold: setiap baris train diprediksi oleh model yang
    TIDAK pernah melihat baris itu. fit_predict(tr, te) melatih model baru
    pada indeks `tr` dan mengembalikan prediksi untuk indeks `te`."""
    out = np.zeros(len(train_idx))
    for tr, te in KFold(n_splits, shuffle=True, random_state=seed).split(train_idx):
        out[te] = fit_predict(train_idx[tr], train_idx[te])
    return out


def fit_fusion(cf_oof, sent_train, y_train, max_depth=10, seed=0):
    """Pohon keputusan di atas [prediksi CF out-of-fold | blok sentimen].
    Blok sentimen: 1 kolom (global/mean), K (concat) atau 2K (concat+conf)."""
    X = np.column_stack([cf_oof, sent_train])
    tree = DecisionTreeRegressor(max_depth=max_depth, random_state=seed).fit(X, y_train)
    return lambda cf_te, sent_te: tree.predict(np.column_stack([cf_te, sent_te]))

## 2. Kode dari buku: uji berpasangan (Kode 3.8.1)

In [2]:
def rmse_gain_pct(se_base, se_new):
    """Penurunan RMSE relatif (%) ; se_* = galat kuadrat per sampel uji."""
    return 100 * (1 - np.sqrt(se_new.mean()) / np.sqrt(se_base.mean()))


def paired_bootstrap_ci(se_base, se_new, n_boot=2000, alpha=0.05, seed=0):
    """CI persentil untuk penurunan RMSE; sampel uji diacak ULANG BERPASANGAN."""
    rng = np.random.default_rng(seed)
    n = len(se_base)
    gains = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        gains[b] = rmse_gain_pct(se_base[idx], se_new[idx])
    return np.percentile(gains, [100 * alpha / 2, 100 * (1 - alpha / 2)])


def per_seed_summary(se_base_by_seed, se_new_by_seed):
    """Wilcoxon per seed (pada galat kuadrat per sampel) + p gabungan Fisher.
    Semua seed memakai set uji yang sama, jadi p gabungan hanyalah ringkasan."""
    p = [stats.wilcoxon(b, n).pvalue for b, n in zip(se_base_by_seed, se_new_by_seed)]
    fisher = stats.combine_pvalues(p, method="fisher").pvalue
    return {"p_per_seed": p, "n_sig": int(sum(x < 0.05 for x in p)), "p_fisher": fisher}

## 3. Data simulasi dan MF dasar

Kelas `AspectMF` sama seperti pada notebook 02. Di sini ia dipakai tanpa suku aspek (`use_aspect=False`) sebagai model CF dasar.

In [3]:
class AspectMF:
    """r_hat = mu + b_u + b_i + p_u.q_i + sum_k a_u^k * s_ui^k  [Pers. 3-6]"""

    def __init__(self, n_users, n_items, n_aspects, d=8, lr=0.02, reg=0.02,
                 use_aspect=True, seed=0):
        rng = np.random.default_rng(seed)
        self.P = rng.normal(0, 0.1, (n_users, d))
        self.Q = rng.normal(0, 0.1, (n_items, d))
        self.bu, self.bi = np.zeros(n_users), np.zeros(n_items)
        self.A = np.zeros((n_users, n_aspects))
        self.mu, self.lr, self.reg = 0.0, lr, reg
        self.use_aspect, self.rng = use_aspect, rng

    def predict(self, u, i, s):
        out = self.mu + self.bu[u] + self.bi[i] + (self.P[u] * self.Q[i]).sum(1)
        return out + (self.A[u] * s).sum(1) if self.use_aspect else out

    def fit(self, u, i, r, s, epochs=40, batch=256):
        self.mu = r.mean()
        lr, lam = self.lr, self.reg
        for _ in range(epochs):
            for idx in np.array_split(self.rng.permutation(len(r)),
                                      max(1, len(r) // batch)):
                ub, ib, sb = u[idx], i[idx], s[idx]
                e = (r[idx] - self.predict(ub, ib, sb))[:, None]  # galat
                pu, qi = self.P[ub].copy(), self.Q[ib].copy()
                np.add.at(self.bu, ub, lr * (e[:, 0] - lam * self.bu[ub]))
                np.add.at(self.bi, ib, lr * (e[:, 0] - lam * self.bi[ib]))
                np.add.at(self.P, ub, lr * (e * qi - lam * pu))
                np.add.at(self.Q, ib, lr * (e * pu - lam * qi))
                if self.use_aspect:  # d r_hat / d a_u^k = s_ui^k
                    np.add.at(self.A, ub, lr * (e * sb - lam * self.A[ub]))
        return self

In [4]:
def simulasi_ulasan(m=600, n=200, K=4, density=0.15, noise=0.3, conf_informatif=True, seed=0):
    """Data ulasan tiruan: rating bergantung pada kualitas aspek item dan bobot aspek pengguna.
    Asumsi yang dibuat (bukan fakta empiris):
      - aspek berbeda punya kepentingan umum berbeda (w_global);
      - sentimen aspek yang DIEKSTRAK = sentimen sebenarnya + derau;
      - konfidens `c` bersifat informatif bila ia menurun saat derau naik (conf_informatif=True),
        atau acak murni (False)."""
    rng = np.random.default_rng(seed)
    mask = rng.random((m, n)) < density
    u, i = np.nonzero(mask)
    N = len(u)
    w_global = np.array([1.6, 0.9, 0.5, 0.2])[:K]           # pentingnya aspek secara umum
    a = rng.dirichlet(w_global * 6, m) * 3.6                 # bobot aspek tiap pengguna
    q = rng.normal(0, 1, (n, K))                             # kualitas aspek tiap item
    t = 1 / (1 + np.exp(-(q[i] + rng.normal(0, .5, (N, K)))))  # sentimen sebenarnya di (0, 1)
    bu, bi = rng.normal(0, .3, m), rng.normal(0, .3, n)
    P, Q = rng.normal(0, 1, (m, 3)), rng.normal(0, 1, (n, 3))
    r = (3.0 + bu[u] + bi[i] + 0.25 * (P[u] * Q[i]).sum(1)
         + (a[u] * (2 * t - 1)).sum(1) + rng.normal(0, .4, N))
    r = np.clip(np.round(r), 1, 5)
    sigma = noise * rng.uniform(0.3, 1.7, (N, K))            # derau ekstraksi bervariasi per sampel
    s = np.clip(t + rng.normal(0, sigma), 0, 1)              # sentimen terekstrak
    if conf_informatif:
        c = np.clip(0.97 - 1.4 * sigma + rng.normal(0, .05, (N, K)), 0.2, 0.97)
    else:
        c = rng.uniform(0.2, 0.97, (N, K))
    return dict(u=u, i=i, r=r, s=s, c=c, a=a, m=m, n=n, K=K)

def rmse(y, yh):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yh)) ** 2)))

def bagi(N, seed, frac=0.2):
    perm = np.random.default_rng(seed).permutation(N)
    return perm[int(frac * N):], perm[:int(frac * N)]        # indeks latih, uji

In [5]:
def cf_dasar(D, tr, te, seed):
    """Prediksi CF (tanpa sentimen): OOF untuk baris latih, dan penuh untuk baris uji."""
    u, i, r, s = D["u"], D["i"], D["r"], D["s"]
    def fit_predict(a, b):
        m = AspectMF(D["m"], D["n"], D["K"], use_aspect=False, seed=seed).fit(u[a], i[a], r[a], s[a])
        return m.predict(u[b], i[b], None)
    cf_oof = oof_predictions(fit_predict, tr, seed=seed)
    m_full = AspectMF(D["m"], D["n"], D["K"], use_aspect=False, seed=seed).fit(u[tr], i[tr], r[tr], s[tr])
    return cf_oof, m_full.predict(u[tr], i[tr], None), m_full.predict(u[te], i[te], None)

def representasi(D):
    return {"mean": D["s"].mean(1, keepdims=True),
            "concat": D["s"],
            "concat_conf": np.column_stack([D["s"], D["c"]])}

D = simulasi_ulasan(noise=0.1, seed=0)
tr, te = bagi(len(D["r"]), 0)
cf_oof, cf_in, cf_te = cf_dasar(D, tr, te, seed=0)
y_te = D["r"][te]
print("RMSE CF saja (uji):", round(rmse(y_te, cf_te), 3))
print("RMSE CF pada data latih (in-sample): %.3f | out-of-fold: %.3f"
      % (rmse(D["r"][tr], cf_in), rmse(D["r"][tr], cf_oof)))

RMSE CF saja (uji): 0.897
RMSE CF pada data latih (in-sample): 0.488 | out-of-fold: 0.930


Perhatikan: prediksi *in-sample* jauh lebih akurat daripada *out-of-fold* karena model sudah melihat baris itu. Jika pohon fusi dilatih pada prediksi in-sample, ia belajar mempercayai CF terlalu banyak.

### Eksperimen A: out-of-fold versus in-sample

Latih pohon fusi dengan dua jenis fitur CF, lalu uji pada data uji yang sama.

In [6]:
rep = representasi(D)
rows = []
for nama, X in rep.items():
    f_oof = fit_fusion(cf_oof, X[tr], D["r"][tr], max_depth=5)
    f_in = fit_fusion(cf_in, X[tr], D["r"][tr], max_depth=5)
    rows.append({"representasi": nama,
                 "RMSE (fitur CF out-of-fold)": rmse(y_te, f_oof(cf_te, X[te])),
                 "RMSE (fitur CF in-sample)": rmse(y_te, f_in(cf_te, X[te]))})
bocor = pd.DataFrame(rows).set_index("representasi")
print(bocor.round(3))
assert (bocor["RMSE (fitur CF out-of-fold)"] < bocor["RMSE (fitur CF in-sample)"]).all()

              RMSE (fitur CF out-of-fold)  RMSE (fitur CF in-sample)
representasi                                                        
mean                                0.827                      0.901
concat                              0.804                      0.893
concat_conf                         0.804                      0.893


**Interpretasi.** Fitur CF out-of-fold menghasilkan RMSE uji yang lebih baik untuk ketiga representasi. Pada fitur in-sample, distribusi fitur saat pelatihan (terlalu akurat) berbeda dengan saat uji (kurang akurat): ini bentuk kebocoran. Skema lipatan menjaga distribusi keduanya sama.

### Eksperimen B: representasi sentimen dan kedalaman pohon

Rata-rata tiga seed pada derau ekstraksi 0,1 dan 0,3. Kedalaman pohon 3, 5, dan 10 (nilai awal di buku).

In [7]:
def eval_grid(noise, depths=(3, 5, 10), seeds=(0, 1, 2)):
    hasil = {}
    for sd in seeds:
        Dn = simulasi_ulasan(noise=noise, seed=sd)
        trn, ten = bagi(len(Dn["r"]), sd)
        co, ci, ct = cf_dasar(Dn, trn, ten, seed=sd)
        hasil.setdefault("CF saja", []).append(rmse(Dn["r"][ten], ct))
        for nama, X in representasi(Dn).items():
            for dep in depths:
                f = fit_fusion(co, X[trn], Dn["r"][trn], max_depth=dep, seed=sd)
                hasil.setdefault((nama, dep), []).append(rmse(Dn["r"][ten], f(ct, X[ten])))
    return {k: float(np.mean(v)) for k, v in hasil.items()}

semua = {noise: eval_grid(noise) for noise in (0.1, 0.3)}
for noise, h in semua.items():
    t = pd.Series({k: v for k, v in h.items() if k != "CF saja"}).unstack()
    t.index.name = "representasi"; t.columns.name = "max_depth"
    print(f"noise = {noise} | RMSE CF saja = {h['CF saja']:.3f}")
    print(t.round(3), "\n")

noise = 0.1 | RMSE CF saja = 0.904
max_depth        3      5      10
representasi                     
concat        0.851  0.812  0.877
concat_conf   0.851  0.812  0.888
mean          0.860  0.828  0.874 

noise = 0.3 | RMSE CF saja = 0.904
max_depth        3      5      10
representasi                     
concat        0.883  0.861  0.941
concat_conf   0.883  0.863  0.958
mean          0.883  0.863  0.910 



**Interpretasi (simulasi).**

1. Menambahkan sentimen ke CF menurunkan RMSE dibandingkan CF saja, pada kedalaman yang wajar.
2. **Kedalaman pohon lebih menentukan daripada pilihan representasi.** Pada kedalaman 10 (nilai awal di kode buku) semua varian memburuk karena pohon menghafal derau. Pada simulasi ini, kedalaman sekitar 5 lebih baik. Itu perlu disetel pada data validasi.
3. Selisih antar representasi kecil. `concat` sedikit unggul atas `mean` pada derau rendah, dan blok konfidens tidak menambah akurasi pada kedalaman 3 dan 5. Pada data nyata di buku pola ini berbeda antar domain, jadi jangan menyimpulkan peringkat representasi dari satu simulasi.

### Eksperimen C: uji berpasangan untuk selisih kecil

Bandingkan `concat` dengan `mean` (kedalaman 5, derau 0,1). Data dan set uji tetap, **seed model** divariasikan (inisialisasi MF, lipatan OOF, pohon). Itu setara dengan "sepuluh seed pada set uji yang sama" pada buku, dengan lima seed agar cepat.

In [8]:
D = simulasi_ulasan(noise=0.1, seed=0)
tr, te = bagi(len(D["r"]), 0)
y_te = D["r"][te]
se_mean, se_concat, se_cf = [], [], []
for sd in range(5):
    co, ci, ct = cf_dasar(D, tr, te, seed=sd)
    rep = representasi(D)
    for nama, X, tujuan in (("mean", rep["mean"], se_mean), ("concat", rep["concat"], se_concat)):
        f = fit_fusion(co, X[tr], D["r"][tr], max_depth=5, seed=sd)
        tujuan.append((y_te - f(ct, X[te])) ** 2)
    se_cf.append((y_te - ct) ** 2)

# (1) perbaikan besar: stacking(mean) terhadap CF saja
# (2) perbaikan kecil: concat terhadap mean
for judul, base, new in (("mean terhadap CF saja", se_cf, se_mean), ("concat terhadap mean", se_mean, se_concat)):
    ringkas = per_seed_summary(base, new)
    gain = rmse_gain_pct(np.mean(base, axis=0), np.mean(new, axis=0))
    ci = paired_bootstrap_ci(np.mean(base, axis=0), np.mean(new, axis=0))
    print(f"[{judul}]")
    print(f"  penurunan RMSE (rata-rata seed): {gain:.2f}% | CI 95% bootstrap berpasangan: [{ci[0]:.2f}, {ci[1]:.2f}]")
    print(f"  Wilcoxon p per seed: {[f'{p:.3g}' for p in ringkas['p_per_seed']]}")
    print(f"  seed signifikan (p < 0,05): {ringkas['n_sig']}/5 | p gabungan Fisher: {ringkas['p_fisher']:.3g}")

[mean terhadap CF saja]
  penurunan RMSE (rata-rata seed): 8.14% | CI 95% bootstrap berpasangan: [6.93, 9.29]
  Wilcoxon p per seed: ['2.22e-15', '2.57e-16', '6.05e-14', '6.91e-18', '4.86e-16']
  seed signifikan (p < 0,05): 5/5 | p gabungan Fisher: 4.64e-69
[concat terhadap mean]
  penurunan RMSE (rata-rata seed): 2.20% | CI 95% bootstrap berpasangan: [0.83, 3.56]
  Wilcoxon p per seed: ['0.0174', '0.042', '0.42', '0.0106', '0.00569']
  seed signifikan (p < 0,05): 4/5 | p gabungan Fisher: 9.82e-05


**Cara membaca.**

- Perbaikan besar (mean terhadap CF saja) jelas: CI jauh dari nol dan semua seed signifikan.
- Perbaikan kecil (concat terhadap mean) perlu hati-hati. Lihat apakah CI melewati nol dan berapa seed yang signifikan. Hasil bisa bergeser antar seed. Karena semua seed memakai **set uji yang sama**, p gabungan Fisher hanya merangkum, dan **bukan** bukti independen: p yang sangat kecil di sana tidak berarti efeknya besar.
- Signifikan secara statistik tidak sama dengan bermakna secara praktis. Laporkan besar efek (persen penurunan RMSE) bersama CI.

## Pertanyaan refleksi

1. Mengapa `fit_predict` harus melatih model *baru* pada setiap lipatan, bukan memakai ulang model yang sama?
2. Pada Eksperimen C, apa yang berubah jika set uji dibangkitkan ulang pada tiap seed? Mengapa p gabungan Fisher lebih sah dalam kasus itu?
3. Tambahkan `min_samples_leaf` pada `DecisionTreeRegressor`. Apakah ia menggantikan penyetelan `max_depth`?